In [62]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from huggingface_hub import InferenceClient



from retrieval_pipeline import *
cfg = load_config()


In [63]:
cfg['hf_model']

'deepseek-ai/DeepSeek-V4-Pro'

In [55]:
vstore = get_vector_store(cfg,)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3772.61it/s]


In [64]:
client = InferenceClient(api_key=cfg['hf_token'] if cfg.get('hf_token') else None)

In [65]:
sample_query = 'How do I integrate slack with cloudedesk?'
docs = retrieve_docs(vstore, sample_query, k=3)
context_str, citation_str = format_docs(docs)

In [66]:
print(context_str)

[Hc-101Integrating-Clouddesk-With-Slack (help_center_articles)]
Integrating CloudDesk with Slack
Doc ID: HC-101  |  Source type: help_center_article  |  Title: Integrating CloudDesk with Slack  |  Last updated: 2026-06-18  | 
Version: v3.6
CloudDesk can post real-time notifications to Slack whenever a ticket is created, updated, or replied to —
keeping your team in the loop without needing to check the dashboard.
Step 1: Connect your Slack workspace
Go to Settings → Integrations → Slack → Connect. You'll be redirected to Slack to authorize CloudDesk.
Approve

---

[Ticket tkt_10021 (support_ticket)]
Ticket [tkt_10021] subject: Slack messages not posting to channel
Question: We connected CloudDesk to Slack but new tickets aren't showing up in #support-alerts.
Resolution: Confirmed the OAuth token had an expired scope after a Slack workspace admin change. Reconnected the integration under Settings > Integrations > Slack and re-selected the channel; messages resumed within a minute.

---


In [67]:
print(citation_str)

- **[Hc-101Integrating-Clouddesk-With-Slack]** (`help_center_articles`) — Similarity: 58%
- **[Ticket tkt_10021]** (`support_ticket`) — Similarity: 45%
- **[Ticket tkt_10023]** (`support_ticket`) — Similarity: 44%


In [72]:
sample_query = 'How do I integrate slack with clouddesk?'
res = run_rag_pipeline(cfg, vstore, client, sample_query)

print(f'Query: {sample_query}')
print(f'Confidence: {res["confidence_pct"]}')
print(f'Escalated: {res["requires_escalation"]}')
print('----- Answer Output ----')
print(res['answer'])

Query: How do I integrate slack with clouddesk?
Confidence: 70%
Escalated: False
----- Answer Output ----
To integrate Slack with CloudDesk, follow these steps:

1.  Navigate to **Settings → Integrations → Slack** in your CloudDesk dashboard.
2.  Click the **Connect** button.
3.  You will be redirected to Slack to authorize the CloudDesk application. Approve the request to complete the connection.

Once connected, CloudDesk will post real-time notifications to your selected Slack channel for ticket events like creation, updates, and replies.

**Troubleshooting Tip:** If notifications stop appearing in your Slack channel (e.g., #support-alerts), the OAuth token may have expired, often due to a Slack workspace admin change. To fix this, simply reconnect the integration by repeating the steps above and re-selecting the target channel. Messages should resume within a minute.

*Sources: [Hc-101Integrating-Clouddesk-With-Slack], [Ticket tkt_10021]*


In [73]:
sample_query = 'How do I cook beans?'
res = run_rag_pipeline(cfg, vstore, client, sample_query)

print(f'Query: {sample_query}')
print(f'Confidence: {res["confidence_pct"]}')
print(f'Escalated: {res["requires_escalation"]}')
print('----- Answer Output ----')
print(res['answer'])

Query: How do I cook beans?
Confidence: 14%
Escalated: True
----- Answer Output ----
I'm sorry, but I cannot answer your question about cooking beans based on the provided context. The context only contains information about the CloudDesk REST API, including customer endpoints and authentication methods. It does not contain any information related to cooking or recipes.

Please contact Tier-2 Support if you have questions about a different topic, or rephrase your question if it relates to the CloudDesk API.


In [74]:
metrics = evaluate_benchmark(cfg, vstore, client)
df_results = pd.DataFrame(metrics['records'])

display(df_results)

[*] Running RAG Evaluation Benchmark across 43 historical tickets...
[*] HuggingFace API notice: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6ab542f3-356ce6af3da39a5c164aff91;59622b67-9713-4499-9510-da464162acd7)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.
[*] HuggingFace API notice: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6ab542f4-6b29663e61ee15113ce464fa;0627aaeb-b9f4-46dc-aea4-68fb0473d77e)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to

,ticket_id,question,ref_title,top_retrieved,confidence,escalated,hit_at_1,hit_at_3
0,tkt_10021,We connected CloudDesk to Slack but new ticket...,Integrating CloudDesk with Slack,Ticket tkt_10021,0.78,False,False,False
1,tkt_10022,Can different ticket categories go to differen...,Integrating CloudDesk with Slack,Ticket tkt_10022,0.75,False,False,False
2,tkt_10023,"After enabling SSO for our workspace, the Slac...",Integrating CloudDesk with Slack,Ticket tkt_10023,0.79,False,False,False
3,tkt_10031,Our webhook receiver is getting events hours a...,Webhooks Reference,Ticket tkt_10031,0.74,False,False,False
4,tkt_10032,Every webhook we receive fails HMAC signature ...,Webhooks Reference,Ticket tkt_10032,0.83,False,False,False
5,tkt_10033,Our webhook endpoint stopped receiving anythin...,Webhook Delivery Failure Response,Ticket tkt_10033,0.75,False,False,True
6,tkt_10034,We added a second domain to our workspace last...,SSO/SAML Authentication Outage Response,Ticket tkt_10034,0.72,False,False,True
7,tkt_10041,Getting 'saml_signature_invalid' for all users...,SAML & SCIM Configuration Reference,Ticket tkt_10041,0.69,False,False,True
8,tkt_10042,We verified our new domain but users on it can...,SAML & SCIM Configuration Reference,Ticket tkt_10042,0.64,False,False,False
9,tkt_10043,Provisioning a new user via SCIM throws scim_d...,SAML & SCIM Configuration Reference,Runbook: SSO/SAML Authentication Outage Response,0.81,False,False,True


In [75]:
metrics['hit_rate_at_1_pct']
metrics['hit_rate_at_3_pct']
metrics['mean_confidence']
metrics['escalation_rate_pct']

'4.65%'

In [78]:
metrics['hit_rate_at_1_pct']

'2.33%'

In [76]:
metrics['hit_rate_at_3_pct']

'51.16%'

In [77]:
metrics['hit_rate_at_5_pct']

'51.16%'

In [79]:
metrics['mean_confidence']


0.7214